In [3]:
%%capture
!pip install -q -U "transformers>=4.46" accelerate peft bitsandbytes trl datasets scikit-learn joblib


In [4]:
import os
import json
import random
import zipfile

import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"


In [8]:
DATA_DIR = "/kaggle/input/datasets/kunimsmk/scolar-red/"

kid_adult_path = os.path.join(DATA_DIR, "kid_adult.jsonl")
public_test_style_path = os.path.join(DATA_DIR, "public_test_style.jsonl")
style_clf_path = os.path.join(DATA_DIR, "style_clf.pkl")

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

kid_adult = load_jsonl(kid_adult_path)
public_test_style = load_jsonl(public_test_style_path)


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(record):
    messages = [
        {"role": "user", "content": record["prompt"]},
        {"role": "assistant", "content": record["kid"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

sft_records = [format_example(r) for r in kid_adult]
train_dataset = Dataset.from_list(sft_records)


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [12]:
sft_config = SFTConfig(
    output_dir="./sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    report_to=[],
    dataset_text_field="text",
    packing=False,
    loss_type="nll"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.321461
20,1.509144
30,1.357944
40,1.229427
50,1.216867
60,1.172698
70,1.178717
80,1.153821
90,1.165350
100,1.021640


TrainOutput(global_step=282, training_loss=1.0080051265709788, metrics={'train_runtime': 3465.747, 'train_samples_per_second': 1.289, 'train_steps_per_second': 0.081, 'total_flos': 1.459239016026624e+16, 'train_loss': 1.0080051265709788, 'entropy': 0.7746880915429857, 'mean_token_accuracy': 0.7879313561651442, 'num_tokens': 612777.0, 'epoch': 3.0})

In [13]:
ADAPTER_DIR = "./sft_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)


('./sft_adapter/tokenizer_config.json',
 './sft_adapter/chat_template.jinja',
 './sft_adapter/tokenizer.json')

In [14]:
model.eval()
model.config.use_cache = True

def generate_response(prompt_text, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [r["prompt"] for r in public_test_style]
generated_responses = [generate_response(p) for p in test_prompts]


In [15]:
import joblib
from scipy.sparse import hstack

style_clf = joblib.load(style_clf_path)
vec1, vec2 = style_clf["vecs"]
estimator = style_clf["clf"]

X = hstack([vec1.transform(generated_responses), vec2.transform(generated_responses)]).tocsr()
p_simple_scores = estimator.predict_proba(X)[:, 1]
p_simple_mean = float(np.mean(p_simple_scores))

print(f"P_simple: {p_simple_mean}")


P_simple: 0.967434839568921


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.9.0. This might lead to breaking c

In [17]:
from trl import DPOTrainer, DPOConfig

def build_dpo_example(record):
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": record["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt_text, "chosen": record["kid"], "rejected": record["adult"]}

dpo_dataset = Dataset.from_list([build_dpo_example(r) for r in kid_adult])


In [18]:
model.config.use_cache = False

dpo_config = DPOConfig(
    output_dir="./dpo_style_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    beta=0.1,
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

dpo_trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Step,Training Loss
10,0.317918
20,0.000399
30,0.000024
40,0.000023
50,0.000010
60,0.000027
70,0.000004
80,0.000001
90,0.000053
100,0.000000


TrainOutput(global_step=282, training_loss=0.011305757738868135, metrics={'train_runtime': 8089.0564, 'train_samples_per_second': 0.552, 'train_steps_per_second': 0.035, 'total_flos': 3.577520376672461e+16, 'train_loss': 0.011305757738868135, 'entropy': 0.7880600425932143, 'num_tokens': 1342536.0, 'logits/chosen': -2.591239033668513, 'logits/rejected': -2.81960731528927, 'mean_token_accuracy': 0.6616268091731601, 'rewards/chosen': -4.919421354929606, 'rewards/rejected': -31.09302732679579, 'rewards/accuracies': 1.0, 'rewards/margins': 26.17360560099284, 'logps/chosen': -115.03189256456163, 'logps/rejected': -487.2759433322483, 'epoch': 3.0})

In [21]:
DPO_STYLE_ADAPTER_DIR = "./dpo_style_adapter"
dpo_trainer.model.save_pretrained(DPO_STYLE_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_STYLE_ADAPTER_DIR)


('./dpo_style_adapter/tokenizer_config.json',
 './dpo_style_adapter/chat_template.jinja',
 './dpo_style_adapter/tokenizer.json')

In [20]:
model.eval()
model.config.use_cache = True

generated_responses = [generate_response(p) for p in test_prompts]

X = hstack([vec1.transform(generated_responses), vec2.transform(generated_responses)]).tocsr()
p_simple_scores = estimator.predict_proba(X)[:, 1]
p_simple_mean = float(np.mean(p_simple_scores))

print(f"P_simple: {p_simple_mean:}")

P_simple: 0.9968234645868576
